In [ ]:
# Analyze attention by sector using BBG_SECTORS from settings
from settings.default import BBG_SECTORS

# Assign sectors to tickers
ticker_sectors = [BBG_SECTORS.get(t, 'Other') for t in ALL_TICKERS]
unique_sectors = sorted(list(set(ticker_sectors)))
print(f"Sectors found ({len(unique_sectors)}): {unique_sectors}")

# Count tickers per sector
sector_counts_dict = {}
for s in ticker_sectors:
    sector_counts_dict[s] = sector_counts_dict.get(s, 0) + 1
print("\nTickers per sector:")
for sector in unique_sectors:
    print(f"  {sector}: {sector_counts_dict[sector]}")

if attention_weights and len(unique_sectors) > 1:
    first_layer = list(attention_weights.keys())[0]
    avg_attn = attention_weights[first_layer]['averaged']
    
    # Compute sector-level attention matrix
    n_sectors = len(unique_sectors)
    sector_attn = np.zeros((n_sectors, n_sectors))
    sector_counts = np.zeros((n_sectors, n_sectors))
    
    for i, src_sector in enumerate(ticker_sectors):
        for j, tgt_sector in enumerate(ticker_sectors):
            src_idx = unique_sectors.index(src_sector)
            tgt_idx = unique_sectors.index(tgt_sector)
            sector_attn[src_idx, tgt_idx] += avg_attn[i, j]
            sector_counts[src_idx, tgt_idx] += 1
    
    # Average attention per sector pair
    sector_attn_avg = sector_attn / np.maximum(sector_counts, 1)
    
    # Plot sector-level attention
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(sector_attn_avg, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(n_sectors))
    ax.set_yticks(range(n_sectors))
    ax.set_xticklabels(unique_sectors, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(unique_sectors, fontsize=9)
    ax.set_xlabel('Target Sector')
    ax.set_ylabel('Source Sector')
    ax.set_title('Average Attention by Sector (Bloomberg Classification)')
    
    # Add text annotations
    for i in range(n_sectors):
        for j in range(n_sectors):
            text = ax.text(j, i, f'{sector_attn_avg[i, j]:.3f}',
                          ha='center', va='center', color='black', fontsize=8)
    
    plt.colorbar(im, label='Avg Attention')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'attention_by_sector.png'), dpi=150)
    plt.show()
    
    # Analyze cross-sector vs intra-sector attention
    intra_sector_attn = []
    cross_sector_attn = []
    for i in range(n_sectors):
        for j in range(n_sectors):
            if i == j:
                intra_sector_attn.append(sector_attn_avg[i, j])
            else:
                cross_sector_attn.append(sector_attn_avg[i, j])
    
    print(f"\nIntra-sector attention (same sector): {np.mean(intra_sector_attn):.4f}")
    print(f"Cross-sector attention (different sectors): {np.mean(cross_sector_attn):.4f}")
    print(f"Ratio (intra/cross): {np.mean(intra_sector_attn)/np.mean(cross_sector_attn):.2f}x")
    
    print("\nSector attention matrix saved!")

In [ ]:
# Find top attended pairs (excluding self-attention)
if attention_weights:
    first_layer = list(attention_weights.keys())[0]
    avg_attn = attention_weights[first_layer]['averaged']
    
    # Remove diagonal (self-attention)
    attn_no_diag = avg_attn.copy()
    np.fill_diagonal(attn_no_diag, 0)
    
    # Get top pairs
    n_top = 20
    flat_indices = np.argsort(attn_no_diag.flatten())[::-1][:n_top]
    
    print(f"\nTop {n_top} Attention Pairs (excluding self-attention):")
    print("="*60)
    print(f"{'Rank':<6} {'Source':<10} {'Target':<10} {'Attention':>12}")
    print("-"*60)
    
    top_pairs = []
    for rank, idx in enumerate(flat_indices, 1):
        i, j = divmod(idx, len(ALL_TICKERS))
        source = ALL_TICKERS[i]
        target = ALL_TICKERS[j]
        weight = attn_no_diag[i, j]
        print(f"{rank:<6} {source:<10} {target:<10} {weight:>12.4f}")
        top_pairs.append((source, target, weight))
    
    # Save top pairs
    top_pairs_df = pd.DataFrame(top_pairs, columns=['Source', 'Target', 'Attention'])
    top_pairs_df.to_csv(os.path.join(output_dir, 'top_attention_pairs.csv'), index=False)
    print(f"\nTop pairs saved to {output_dir}/top_attention_pairs.csv")

In [ ]:
# Compare learned attention with original graph structure
original_graph = pd.read_csv(GRAPH_FILE, index_col=0)
original_graph = original_graph.reindex(index=ALL_TICKERS, columns=ALL_TICKERS, fill_value=0)
original_adj = original_graph.values

if attention_weights:
    first_layer = list(attention_weights.keys())[0]
    learned_attn = attention_weights[first_layer]['averaged']
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Original graph
    im1 = axes[0].imshow(original_adj, cmap='Blues', aspect='auto')
    axes[0].set_title('Original Graph (CVX/Pearson)')
    axes[0].set_xlabel('Target')
    axes[0].set_ylabel('Source')
    plt.colorbar(im1, ax=axes[0], label='Edge Weight')
    
    # Learned attention (masked by original graph)
    masked_attn = learned_attn * (original_adj > 0).astype(float)
    im2 = axes[1].imshow(masked_attn, cmap='YlOrRd', aspect='auto')
    axes[1].set_title('Learned Attention (on graph edges)')
    axes[1].set_xlabel('Target')
    axes[1].set_ylabel('Source')
    plt.colorbar(im2, ax=axes[1], label='Attention Weight')
    
    plt.suptitle('Original Graph vs Learned Attention', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'graph_vs_attention.png'), dpi=150)
    plt.show()
    
    # Correlation between original edge weights and learned attention
    mask = original_adj > 0
    if mask.sum() > 0:
        orig_edges = original_adj[mask]
        learned_edges = learned_attn[mask]
        correlation = np.corrcoef(orig_edges, learned_edges)[0, 1]
        print(f"\nCorrelation between original edge weights and learned attention: {correlation:.4f}")

In [ ]:
# Plot attention weights per head
if attention_weights:
    first_layer = list(attention_weights.keys())[0]
    head_attns = attention_weights[first_layer]['per_head']
    num_heads = len(head_attns)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    axes = axes.flatten()
    
    for i, (ax, head_attn) in enumerate(zip(axes, head_attns)):
        im = ax.imshow(head_attn, cmap='YlOrRd', aspect='auto')
        ax.set_title(f'Head {i+1}')
        ax.set_xlabel('Target')
        ax.set_ylabel('Source')
        plt.colorbar(im, ax=ax)
    
    plt.suptitle(f'Attention Weights by Head ({first_layer})', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'attention_per_head.png'), dpi=150)
    plt.show()

In [ ]:
# Visualize attention weights as heatmap
from settings.default import ALL_TICKERS

def plot_attention_heatmap(attn_matrix, title, tickers, ax=None, show_labels=True):
    """Plot attention weights as a heatmap."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 12))
    
    im = ax.imshow(attn_matrix, cmap='YlOrRd', aspect='auto')
    
    if show_labels and len(tickers) <= 50:
        ax.set_xticks(range(len(tickers)))
        ax.set_yticks(range(len(tickers)))
        ax.set_xticklabels(tickers, rotation=90, fontsize=6)
        ax.set_yticklabels(tickers, fontsize=6)
    
    ax.set_xlabel('Target (attending to)')
    ax.set_ylabel('Source (attending from)')
    ax.set_title(title)
    
    plt.colorbar(im, ax=ax, label='Attention Weight')
    
    return ax

# Plot averaged attention for first GAT layer
if attention_weights:
    first_layer = list(attention_weights.keys())[0]
    avg_attn = attention_weights[first_layer]['averaged']
    
    fig, ax = plt.subplots(figsize=(14, 12))
    plot_attention_heatmap(avg_attn, f'Averaged Attention Weights ({first_layer})', ALL_TICKERS, ax)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'attention_heatmap.png'), dpi=150)
    plt.show()
    
    print(f"\nAttention statistics:")
    print(f"  Mean: {avg_attn.mean():.4f}")
    print(f"  Std:  {avg_attn.std():.4f}")
    print(f"  Max:  {avg_attn.max():.4f}")
    print(f"  Min:  {avg_attn.min():.4f}")

In [ ]:
import tensorflow as tf
from tensorflow import keras

def extract_attention_weights(model, sample_input):
    """
    Extract attention weights from GAT layers by creating intermediate models.
    
    Returns attention weights for each GAT layer and each attention head.
    """
    # Find GAT layers in the model
    gat_layers = [layer for layer in model.layers if 'gat' in layer.name.lower()]
    print(f"Found {len(gat_layers)} GAT layers: {[l.name for l in gat_layers]}")
    
    attention_weights_all = {}
    
    for gat_layer in gat_layers:
        # Get the layer's weights
        W = gat_layer.W.numpy()  # (heads, input_dim, units)
        a_src = gat_layer.a_src.numpy()  # (heads, units, 1)
        a_dst = gat_layer.a_dst.numpy()  # (heads, units, 1)
        attn_heads = gat_layer.attn_heads
        
        # Get input to this layer by creating intermediate model
        # Find the layer that feeds into this GAT layer
        layer_input = None
        for i, layer in enumerate(model.layers):
            if layer.name == gat_layer.name:
                # Get previous layer's output
                if i > 0:
                    intermediate_model = keras.Model(
                        inputs=model.input,
                        outputs=model.layers[i-1].output if hasattr(model.layers[i-1], 'output') else model.layers[i].input
                    )
                    layer_input = intermediate_model.predict(sample_input[:1], verbose=0)
                break
        
        if layer_input is None:
            print(f"Could not get input for layer {gat_layer.name}")
            continue
            
        # Reshape if needed (batch, tickers, time, features) -> (batch*time, tickers, features)
        if len(layer_input.shape) == 4:
            batch, time, nodes, feats = layer_input.shape
            layer_input = layer_input.reshape(-1, nodes, feats)
        
        # Compute attention weights for each head
        head_attentions = []
        for head in range(attn_heads):
            # Linear transformation
            h = np.matmul(layer_input, W[head])  # (batch*time, nodes, units)
            
            # Attention scores
            attn_src = np.matmul(h, a_src[head])  # (batch*time, nodes, 1)
            attn_dst = np.matmul(h, a_dst[head])  # (batch*time, nodes, 1)
            
            # Pairwise scores
            attn_scores = attn_src + attn_dst.transpose(0, 2, 1)  # (batch*time, nodes, nodes)
            
            # LeakyReLU
            attn_scores = np.where(attn_scores > 0, attn_scores, 0.2 * attn_scores)
            
            # Apply mask if sparse
            if hasattr(gat_layer, 'attention_mask') and gat_layer.attention_mask is not None:
                mask = gat_layer.attention_mask.numpy()
                attn_scores = np.where(mask > 0, attn_scores, -1e9)
            
            # Softmax
            attn_weights = np.exp(attn_scores - attn_scores.max(axis=-1, keepdims=True))
            attn_weights = attn_weights / attn_weights.sum(axis=-1, keepdims=True)
            
            # Average over batch*time dimension
            avg_attn = attn_weights.mean(axis=0)  # (nodes, nodes)
            head_attentions.append(avg_attn)
        
        attention_weights_all[gat_layer.name] = {
            'per_head': head_attentions,
            'averaged': np.mean(head_attentions, axis=0)
        }
    
    return attention_weights_all

# Extract attention weights using a sample from test data
sample_input = model_features.test_sliding['inputs'][:10]
attention_weights = extract_attention_weights(best_model, sample_input)

print("\nAttention weights extracted!")
for layer_name, weights in attention_weights.items():
    print(f"  {layer_name}: {len(weights['per_head'])} heads, shape {weights['averaged'].shape}")

# LSTM-Sparse-GAT Training & Evaluation

This notebook trains an LSTM with Sparse Graph Attention Network for delta-neutral straddle momentum trading.

**Architecture**: Shared LSTM per ticker → Sparse GAT (attention masked by precomputed graph) → Position output

**Sparse GAT**: Uses the precomputed adjacency matrix as an attention mask. Nodes can only attend to their connected neighbors.

All hyperparameters and graph settings can be configured in Section 1 below.

## 0. Setup & Installation

In [ ]:
# Install dependencies (run on Colab)
import sys
if 'google.colab' in sys.modules:
    !pip install -q tensorflow>=2.16.0 keras-tuner empyrical-reloaded spektral
    print("Dependencies installed!")

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Detect environment and set working directory
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/4YP-main'  # UPDATE THIS PATH
else:
    PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    if not os.path.exists(os.path.join(PROJECT_DIR, 'gml')):
        PROJECT_DIR = '..'  

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Apply compatibility fixes for Keras 3 / NumPy 2.0
import re

def apply_fixes(filepath):
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return
    with open(filepath, 'r') as f:
        content = f.read()
    
    original = content
    
    # Fix np.NINF -> -np.inf
    content = content.replace('np.NINF', '-np.inf')
    
    # Fix get_best_models -> load_model
    content = re.sub(
        r'best_model = self\.tuner\.get_best_models\(num_models=1\)\[0\]',
        'best_model = self.load_model(best_hp)',
        content
    )
    
    # Add restore_best_weights=True if missing
    if 'restore_best_weights=True' not in content:
        content = content.replace(
            'min_delta=1e-4,\n            )\n        ]\n        training_history = best_model.fit',
            'min_delta=1e-4,\n                restore_best_weights=True,\n            )\n        ]\n        training_history = best_model.fit'
        )
    
    if content != original:
        with open(filepath, 'w') as f:
            f.write(content)
        print(f"Fixed: {filepath}")
    else:
        print(f"No changes needed: {filepath}")

apply_fixes('gml/graph_model_2.py')
apply_fixes('gml/deep_neural_network.py')
apply_fixes('gml/graph_model_gat.py')

## 1. Configuration

**Modify the settings below to customize training.**

In [ ]:
#==============================================================================
# TRAINING CONFIGURATION - MODIFY THESE SETTINGS
#==============================================================================

# Experiment name (used for saving results)
EXPERIMENT_NAME = "lstm_sparse_gat_experiment"

# Data split
TRAIN_START = 2011
TEST_START = 2017
TEST_END = 2023

# Feature file path
FEATURES_FILE = "data/straddle_features/features.csv"

#==============================================================================
# GRAPH CONFIGURATION (used as attention mask)
#==============================================================================

# Graph type: "cvx" or "pearson"
GRAPH_TYPE = "cvx"

# CVX graph parameters (only used if GRAPH_TYPE = "cvx")
CVX_ALPHA = 100
CVX_BETA = 0.01

# Pearson graph parameters (only used if GRAPH_TYPE = "pearson")
PEARSON_TAU = 0.45

#==============================================================================
# MODEL HYPERPARAMETERS
#==============================================================================

# LSTM settings
HIDDEN_LAYER_SIZE = 10      # LSTM hidden units
DROPOUT_RATE = 0.4          # Dropout rate
TIME_STEPS = 20             # Lookback window

# GAT settings
GAT_UNITS = 16              # GAT output dimension per head
ATTENTION_HEADS = 4         # Number of attention heads
NUM_GAT_LAYERS = 2          # Number of GAT layers (1 or 2)

# Training settings
LEARNING_RATE = 0.001       # Adam learning rate
MAX_GRADIENT_NORM = 0.01    # Gradient clipping
BATCH_SIZE = 32             # Mini-batch size
NUM_EPOCHS = 300            # Maximum epochs
EARLY_STOPPING_PATIENCE = 25  # Early stopping patience

# Hyperparameter search (set to 1 for single config, >1 for random search)
HP_SEARCH_ITERATIONS = 1

# IMPORTANT: Set to True to force fresh training (deletes cached tuner results)
FORCE_RETRAIN = True

#==============================================================================
print("Configuration loaded!")
print(f"  Experiment: {EXPERIMENT_NAME}")
print(f"  Train: {TRAIN_START}-{TEST_START}, Test: {TEST_START}-{TEST_END}")
print(f"  Graph (attention mask): {GRAPH_TYPE}" + (f" (alpha={CVX_ALPHA}, beta={CVX_BETA})" if GRAPH_TYPE == 'cvx' else f" (tau={PEARSON_TAU})"))
print(f"  LSTM: hidden={HIDDEN_LAYER_SIZE}, dropout={DROPOUT_RATE}")
print(f"  Sparse GAT: units={GAT_UNITS}, heads={ATTENTION_HEADS}, layers={NUM_GAT_LAYERS}")
print(f"  Force retrain: {FORCE_RETRAIN}")

In [ ]:
# Override hp_grid.py settings with notebook configuration
import settings.hp_grid as hp_grid

hp_grid.HP_HIDDEN_LAYER_SIZE_GRAPH = [HIDDEN_LAYER_SIZE]
hp_grid.HP_DROPOUT_RATE_GRAPH = [DROPOUT_RATE]
hp_grid.HP_LEARNING_RATE_GRAPH = [LEARNING_RATE]
hp_grid.HP_MAX_GRADIENT_NORM_GRAPH = [MAX_GRADIENT_NORM]
hp_grid.HP_MINIBATCH_SIZE_GRAPH = [BATCH_SIZE]
hp_grid.HP_GCN_UNITS = [GAT_UNITS]  # Used as gat_units
hp_grid.HP_ALPHA = [CVX_ALPHA]
hp_grid.HP_BETA = [CVX_BETA]
hp_grid.HP_ATTN_HEADS = [ATTENTION_HEADS]

# Override fixed_params
import settings.fixed_params as fixed_params
fixed_params.MODEL_PARAMS_GRAPH['total_time_steps'] = TIME_STEPS
fixed_params.MODEL_PARAMS_GRAPH['num_epochs'] = NUM_EPOCHS
fixed_params.MODEL_PARAMS_GRAPH['early_stopping_patience'] = EARLY_STOPPING_PATIENCE
fixed_params.MODEL_PARAMS_GRAPH['random_search_iterations'] = HP_SEARCH_ITERATIONS

print("Hyperparameters overridden!")

In [ ]:
# Configure graph file path based on selection
if GRAPH_TYPE == "cvx":
    GRAPH_FILE = f"data/graph_structure/cvx_opt/{CVX_ALPHA}_{CVX_BETA}_cvx.csv"
else:
    GRAPH_FILE = f"data/graph_structure/pearson/{PEARSON_TAU}.csv"

print(f"Graph file: {GRAPH_FILE}")

# Verify file exists
if not os.path.exists(GRAPH_FILE):
    print(f"WARNING: Graph file not found!")
else:
    print("Graph file found!")

## 2. Load Data & Prepare Features

In [ ]:
from settings.default import ALL_TICKERS
from settings.fixed_params import MODEL_PARAMS_GRAPH
from gml.graph_model_inputs import GraphModelFeatures

# Load raw data
raw_data = pd.read_csv(FEATURES_FILE, index_col=0, parse_dates=True)
raw_data["date"] = raw_data["date"].astype("datetime64[ns]")

print(f"Data shape: {raw_data.shape}")
print(f"Date range: {raw_data['date'].min()} to {raw_data['date'].max()}")
print(f"Tickers: {raw_data['ticker'].nunique()}")

In [ ]:
# Prepare model parameters
params = MODEL_PARAMS_GRAPH.copy()
params["architecture"] = "LSTM-Sparse-GAT"

# Create model features
model_features = GraphModelFeatures(
    raw_data,
    params["total_time_steps"],
    start_boundary=TRAIN_START,
    test_boundary=TEST_START,
    test_end=TEST_END,
    split_tickers_individually=params["split_tickers_individually"],
    train_valid_ratio=params["train_valid_ratio"],
    time_features=params["time_features"],
    lags=params["force_output_sharpe_length"] if params["force_output_sharpe_length"] else None,
)

print(f"\nNumber of tickers: {model_features.num_tickers}")
print(f"Train samples: {model_features.train['inputs'].shape}")
print(f"Valid samples: {model_features.valid['inputs'].shape}")
print(f"Test samples: {model_features.test_sliding['inputs'].shape}")

## 3. Initialize & Train Model

In [ ]:
# Patch the GAT model to use our selected graph file
filepath = 'gml/graph_model_gat.py'
with open(filepath, 'r') as f:
    content = f.read()

# Replace graph file path in SparseGATLSTMDeepMomentumNetwork
import re
content = re.sub(
    r'graph_file = os\.path\.join\("data", "graph_structure", "cvx_opt", f"\{alpha\}_\{beta\}_cvx\.csv"\)',
    f'graph_file = "{GRAPH_FILE}"',
    content
)
content = re.sub(
    r'graph_file = "data/graph_structure/[^"]+"',
    f'graph_file = "{GRAPH_FILE}"',
    content
)

# Also fix attention heads to use our setting
content = re.sub(
    r'attn_heads = hp\.Choice\("attn_heads", values=\[[0-9, ]+\]\)',
    f'attn_heads = hp.Choice("attn_heads", values=[{ATTENTION_HEADS}])',
    content
)

# Fix number of layers
content = re.sub(
    r'num_gat_layers = hp\.Choice\("gat_layers", values=\[[0-9, ]+\]\)',
    f'num_gat_layers = hp.Choice("gat_layers", values=[{NUM_GAT_LAYERS}])',
    content
)

with open(filepath, 'w') as f:
    f.write(content)

print(f"Graph file set to: {GRAPH_FILE}")
print(f"Attention heads: {ATTENTION_HEADS}")
print(f"GAT layers: {NUM_GAT_LAYERS}")

In [ ]:
# Reload the module after patching
import importlib
import gml.graph_model_gat
importlib.reload(gml.graph_model_gat)
from gml.graph_model_gat import SparseGATLSTMDeepMomentumNetwork

# Create output directory
output_dir = os.path.join("results", EXPERIMENT_NAME, f"{TEST_START}-{TEST_END}")
hp_directory = os.path.join(output_dir, "hp")
os.makedirs(output_dir, exist_ok=True)

# Clear cached tuner if FORCE_RETRAIN is True
if FORCE_RETRAIN and os.path.exists(hp_directory):
    import shutil
    shutil.rmtree(hp_directory)
    print(f"Cleared cached tuner at: {hp_directory}")

# Initialize model
dmn = SparseGATLSTMDeepMomentumNetwork(
    EXPERIMENT_NAME,
    hp_directory,
    hp_minibatch_size=[BATCH_SIZE],
    num_tickers=model_features.num_tickers,
    **params,
    **model_features.input_params,
)

print(f"Sparse GAT Model initialized!")
print(f"Output directory: {output_dir}")

In [ ]:
# Train model
print("Starting training...")
print(f"  Max epochs: {NUM_EPOCHS}")
print(f"  Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"  Batch size: {BATCH_SIZE}")
print()

best_hp, best_model, training_history = dmn.hyperparameter_search(
    model_features.train,
    model_features.valid
)

print("\nTraining complete!")
print("\nBest hyperparameters:")
for k, v in best_hp.items():
    print(f"  {k}: {v}")

## 4. Visualize Training

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(training_history.history['loss'], label='Train Loss', linewidth=2)
if 'val_loss' in training_history.history:
    ax.plot(training_history.history['val_loss'], label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (Negative Sharpe)')
ax.set_title('Sparse GAT Training & Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'loss_plot.png'), dpi=150)
plt.show()

print(f"Epochs trained: {len(training_history.history['loss'])}")
print(f"Final train loss: {training_history.history['loss'][-1]:.4f}")
if 'val_loss' in training_history.history:
    print(f"Final val loss: {training_history.history['val_loss'][-1]:.4f}")

## 5. Evaluate on Test Set

In [ ]:
# Get predictions on test set
test_data = model_features.test_sliding
inputs = test_data['inputs']
outputs = test_data['outputs']
time_data = test_data['date']

positions = best_model.predict(inputs)

# Use only last timestep from each window (correct evaluation)
positions_last = positions[:, :, -1, 0]
outputs_last = outputs[:, :, -1, 0]
time_last = time_data[:, :, -1, 0]

captured = positions_last * outputs_last

# Aggregate by day
df = pd.DataFrame({
    'time': pd.to_datetime(time_last.flatten()),
    'captured': captured.flatten()
})
daily_returns = df.groupby('time')['captured'].mean()

print(f"Test days: {len(daily_returns)}")

In [ ]:
# Calculate performance metrics
annual_ret = daily_returns.mean() * 252
annual_vol = daily_returns.std() * np.sqrt(252)
sharpe = annual_ret / annual_vol
hit_rate = (daily_returns > 0).mean()

# Sortino ratio
downside = daily_returns[daily_returns < 0].std() * np.sqrt(252)
sortino = annual_ret / downside if downside > 0 else np.nan

# Max drawdown
cum_returns = (1 + daily_returns).cumprod()
rolling_max = cum_returns.expanding().max()
drawdown = (cum_returns - rolling_max) / rolling_max
max_dd = drawdown.min()

# Calmar ratio
calmar = annual_ret / abs(max_dd) if max_dd != 0 else np.nan

print("\n" + "="*50)
print("SPARSE GAT PERFORMANCE METRICS")
print("="*50)
print(f"{'Annual Return':20s}: {annual_ret*100:>10.2f}%")
print(f"{'Annual Volatility':20s}: {annual_vol*100:>10.2f}%")
print(f"{'Sharpe Ratio':20s}: {sharpe:>10.2f}")
print(f"{'Sortino Ratio':20s}: {sortino:>10.2f}")
print(f"{'Max Drawdown':20s}: {max_dd*100:>10.2f}%")
print(f"{'Calmar Ratio':20s}: {calmar:>10.2f}")
print(f"{'Hit Rate':20s}: {hit_rate*100:>10.1f}%")
print("="*50)
print(f"\n{'RESCALED TO 15% VOL':20s}: {annual_ret * (0.15 / annual_vol)*100:>10.2f}% return")

In [ ]:
# Position distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(positions_last.flatten(), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0].axvline(x=0, color='r', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Sparse GAT Position Distribution')
axes[0].grid(True, alpha=0.3)

# Add stats
stats_text = f"Mean: {positions_last.mean():.3f}\nStd: {positions_last.std():.3f}\nMin: {positions_last.min():.3f}\nMax: {positions_last.max():.3f}"
axes[0].text(0.95, 0.95, stats_text, transform=axes[0].transAxes, fontsize=10,
             verticalalignment='top', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Daily returns distribution
axes[1].hist(daily_returns.values * 100, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].axvline(x=0, color='r', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Daily Return (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Daily Returns Distribution')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'distributions.png'), dpi=150)
plt.show()

## 6. Cumulative Returns

In [ ]:
# Plot cumulative returns
cumulative_returns = (1 + daily_returns).cumprod() - 1

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(cumulative_returns.index, cumulative_returns.values * 100, linewidth=1.5, color='orange')
ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax.fill_between(cumulative_returns.index, 0, cumulative_returns.values * 100, 
                where=cumulative_returns.values >= 0, alpha=0.3, color='green')
ax.fill_between(cumulative_returns.index, 0, cumulative_returns.values * 100, 
                where=cumulative_returns.values < 0, alpha=0.3, color='red')

ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (%)')
ax.set_title(f'Sparse GAT Cumulative Returns ({TEST_START}-{TEST_END})')
ax.grid(True, alpha=0.3)

# Add final return annotation
final_return = cumulative_returns.iloc[-1] * 100
ax.annotate(f'Final: {final_return:.1f}%', 
            xy=(cumulative_returns.index[-1], final_return),
            xytext=(10, 0), textcoords='offset points',
            fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'cumulative_returns.png'), dpi=150)
plt.show()

## 7. Save Results

In [ ]:
# Save all results
results_df = pd.DataFrame({
    'date': daily_returns.index,
    'daily_return': daily_returns.values,
    'cumulative_return': cumulative_returns.values
})
results_df.to_csv(os.path.join(output_dir, 'daily_returns.csv'), index=False)

# Save metrics
metrics = {
    'annual_return': float(annual_ret),
    'annual_volatility': float(annual_vol),
    'sharpe_ratio': float(sharpe),
    'sortino_ratio': float(sortino),
    'max_drawdown': float(max_dd),
    'calmar_ratio': float(calmar),
    'hit_rate': float(hit_rate),
}
with open(os.path.join(output_dir, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=4)

# Save hyperparameters
with open(os.path.join(output_dir, 'hyperparameters.json'), 'w') as f:
    json.dump(best_hp, f, indent=4)

# Save config
config = {
    'experiment_name': EXPERIMENT_NAME,
    'model_type': 'Sparse GAT',
    'train_start': TRAIN_START,
    'test_start': TEST_START,
    'test_end': TEST_END,
    'graph_type': GRAPH_TYPE,
    'graph_file': GRAPH_FILE,
    'hidden_layer_size': HIDDEN_LAYER_SIZE,
    'dropout_rate': DROPOUT_RATE,
    'gat_units': GAT_UNITS,
    'attention_heads': ATTENTION_HEADS,
    'num_gat_layers': NUM_GAT_LAYERS,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'num_epochs': NUM_EPOCHS,
}
with open(os.path.join(output_dir, 'config.json'), 'w') as f:
    json.dump(config, f, indent=4)

print(f"Results saved to: {output_dir}")
print("\nFiles:")
for f in os.listdir(output_dir):
    print(f"  - {f}")

## 8. Attention Weight Visualization

Extract and visualize the learned attention weights to understand which stocks attend to which others.